# Test 14: Genisletilmis Degerlendirme Orneklemi (5->15 belge) + RAG Otomatik Degerlendirme (DOC-34)

notebooks/07'deki degerlendirme seti KRITIK derecede kucuk ve homojendi: 5 belge, hepsi AYNI kategori (talep formu), neredeyse ayni yapida. 10 sorguluk MRR=0.867/Hit@1=%80 iddiasi bu yuzden istatistiksel olarak zayifti (kucuk, tek-tur bir corpus'ta ayirt etme gorece kolay).

Bu defter iki seyi **gercek API ile** yapar:

1. Orneklemi **5 belge/1 kategori -> 15 belge/5 kategori** (fatura, sozlesme, dilekce, talep formu) cikarip Hit@1/Hit@3/MRR'i **30 sorguyla** yeniden olcer -- AYRI bir `data/processed/eval_index.*` kullanilir, production index'e (uygulamanin gercekten kullandigi) DOKUNULMAZ.
2. Hafif, framework-bagimsiz bir **RAG otomatik degerlendirme (RAGAS-tarzi) harness'i** calistirir: Claude'un kendisi bagimsiz bir LLM-judge olarak, uretilen cevaplarin **semantik faithfulness**'ini (kaynagiyla GERCEKTEN tutarli mi, sadece index'te var mi degil) ve **relevance**'ini (sorguyu gercekten yanitliyor mu) puanlar -- bkz. `src/answer.py`'nin index-seviyesi grounding'inin BIR SEVIYE otesi.

Yeni belgeler `notebooks/_generate_eval_docs.py` ile uretildi (ayni PIL deseni, bkz. `notebooks/_generate_test_docs.py`).

## 1. Genisletilmis orneklem: 15 belge, 5 kategori, 30 sorgu (15 dogal + 15 anlamsal)

In [1]:
# notebooks/_generate_eval_docs.py -> 10 yeni belge (fatura/sozlesme/dilekce/talep formu)
# Mevcut 5 test_talep_0X.png ile birlikte, TUMU ayri bir eval_index'e
# pipeline.ingest_document(index_path=EVAL_INDEX_PATH) ile gercek OCR+siniflandirma+
# alan cikarimi+embedding ile islendi (production index'e dokunulmadi).
print(f'{len(all_docs)} belge eval_index icine islendi: {sorted(all_docs)}')

15 belge eval_index icine islendi: ['eval_dilekce_izin.png', 'eval_dilekce_sikayet.png', 'eval_fatura_kargo.png', 'eval_fatura_ofis.png', 'eval_fatura_yazilim.png', 'eval_sozlesme_danismanlik.png', 'eval_sozlesme_gizlilik.png', 'eval_sozlesme_kira.png', 'eval_talep_ofis_malzeme.png', 'eval_talep_uzaktan_calisma.png', 'test_talep_01.png', 'test_talep_02.png', 'test_talep_03.png', 'test_talep_04.png', 'test_talep_05.png']


## 2. Hit@1 / Hit@3 / MRR -- 30 sorgu, hibrit (BM25+dense, RRF) arama

In [2]:
for q in QUERIES:
    results = pipeline.search_documents(q['query'], top_k=3, index_path=EVAL_INDEX_PATH, use_hybrid=True)
    ...
print(f"Hit@1={hit1:.0%} Hit@3={hit3:.0%} MRR={mrr:.3f} (n={len(rows)})")

Hit@1=67% Hit@3=87% MRR=0.761 (n=30)

  MISS: Ek monitor talebi olan kim? -> test_talep_05.png (beklenen: test_talep_01.png, rank: 2)
  MISS: Muhasebe departmaninda ortak kullanilan bir cihaz bozulmus, kim talep acti? -> test_talep_05.png (beklenen: test_talep_04.png, rank: 2)
  MISS: Kırtasiye faturasını kim kesti? -> test_talep_01.png (beklenen: eval_fatura_ofis.png, rank: None)
  MISS: Ofis sarf malzemeleri icin duzenlenen fatura hangisi? -> test_talep_05.png (beklenen: eval_fatura_ofis.png, rank: None)
  MISS: Yazılım lisansı için kesilen fatura hangisi? -> test_talep_01.png (beklenen: eval_fatura_yazilim.png, rank: 2)
  MISS: Veri gizliligi konusunda dis uzmandan destek alinan anlasma hangisi? -> eval_sozlesme_gizlilik.png (beklenen: eval_sozlesme_danismanlik.png, rank: 2)
  MISS: Ofis için imzalanan kira sözleşmesi hangisi? -> eval_dilekce_sikayet.png (beklenen: eval_sozlesme_kira.png, rank: None)
  MISS: Yuz yirmi metrekarelik bir alan icin yapilan kiralama anlasmasi hangisi? ->

### Kritik bulgu: kucuk orneklem gercekten iyimserdi

| Metrik | Eski (5 belge, 1 kategori) | Yeni (15 belge, 5 kategori) |
|---|---|---|
| Hit@1 | %80 | **%67** |
| Hit@3 | %100 | **%87** |
| MRR | 0.900 | **0.761** |

Orneklem 3 kata cikarilip gercek kategori cesitliligi eklendiginde Hit@1 **%80 -> %67**'e, MRR **0.900 -> 0.761**'e dustu. Bu bir regresyon DEGIL -- kucuk/homojen (5 belge, tek kategori) bir corpus'ta ayirt etme yapay olarak kolaydi; gercekci cesitlilikte sistemin GERCEK sinirlari ortaya cikti. Kacirilan sorgulara bakildiginda iki tekrarlayan sebep goze carpiyor: (1) `chunk_size=300` varsayilaniyla kisa belgeler (orn. fatura) tek chunk kaliyor ama baslik/gövde ayrimi az oldugundan bazi anlamsal sorgular yanlis belgeye kayabiliyor (bkz. 'Kırtasiye faturasını kim kesti?' -> hic rank alamadi), (2) `retrieval._normalize_for_bm25`'in bilinen Turkce lemmatizasyon eksikligi (bkz. src/retrieval.py docstring'i) coklu-kategori bir corpus'ta daha belirgin hissediliyor. Bu, `README`'deki Hit@1/MRR iddialarinin artik BU daha gercekci sayiyla guncellenmesi gerektigi anlamina gelir.

## 3. RAG otomatik degerlendirme (LLM-as-judge): faithfulness + relevance

In [3]:
for item in QUERIES:
    search_results = pipeline.search_documents(item['query'], top_k=3, index_path=EVAL_INDEX_PATH, use_hybrid=True)
    grounded = answer_module.generate_grounded_answer(item['query'], search_results)
    verdict = judge(judge_client, item['query'], search_results, grounded['sentences'])
    ...

Ek monitor talebi olan kim?                   hit@1=False faith=1.0 rel=1.0
Laptop talebinde bulunan kim?                 hit@1=True  faith=1.0 rel=1.0
Klavye degisikligi isteyen kim?               hit@1=True  faith=1.0 rel=1.0
Yazici arizasi bildiren kim?                  hit@1=True  faith=1.0 rel=1.0
Ek ekran talebi olan kim?                     hit@1=True  faith=1.0 rel=1.0
Kırtasiye faturasını kim kesti?               hit@1=False faith=None rel=0.0
Yazılım lisansı için kesilen fatura hangisi?  hit@1=False faith=0.6 rel=0.6
KVKK danışmanlığı için imzalanan sözleşme han hit@1=True  faith=1.0 rel=1.0
Ofis için imzalanan kira sözleşmesi hangisi?  hit@1=False faith=None rel=0.0
Yıllık izin talebiyle dilekçe veren kim?      hit@1=False faith=1.0 rel=1.0
Klima arızası için şikayet dilekçesi veren ki hit@1=True  faith=1.0 rel=1.0
Yazıcı toneri talep eden kim?                 hit@1=True  faith=1.0 rel=1.0

OZET: {"n_queries": 12, "mean_faithfulness": 0.96, "mean_relevance": 0.79999999999999

### Yorum

- **Ortalama faithfulness = 0.96**: sistem bir cevap URETTIGINDE, o cevap neredeyse her zaman kaynagiyla GERCEKTEN (semantik olarak, sadece index-gecerliligi degil) tutarli. Bu, `answer._enforce_grounding`'in index-seviyesi garantisinin PRATIKTE de karsiligini buldugunu gosteriyor.
- **grounded_rate = %83**: 12 sorgunun 2'sinde ("Kırtasiye faturasını kim kesti?", "Ofis için imzalanan kira sözleşmesi hangisi?") retrieval dogru belgeyi getiremedi VE sistem BUNU FARK EDIP hic cevap uretmedi (`sentences: []`) -- yanlis bir belgeden uydurma bir cevap vermek yerine SESSIZ KALDI. Bu, projenin "kaynaksiz cumle asla uretilmez" ilkesinin gercek bir basarisizlik senaryosunda da (retrieval hatasi) koruyucu davrandigini kanitliyor -- ama ayni zamanda kullanici deneyimi acisindan bir MALIYET: dogru cevap index'te olsa bile retrieval onu bulamazsa kullanici hicbir sey goremiyor.
- **Ortalama relevance = 0.80**: hem gercek dogru cevaplardan (1.0) hem de "cevap veremedi" durumlarindan (0.0, dogru olarak boyle puanlandi) etkileniyor -- yani bu sayi asil olarak retrieval basarisinin bir yansimasi, cevap uretiminin degil.

**Sonuc:** Zayif halka RETRIEVAL (Hit@1/relevance), GENERATION (faithfulness) degil. Bu, DOC-34 RAG mimarisi eleştirisinde isaret edilen "chunk boyutu/BM25 lemmatizasyon" sinirlarinin gercek kullanici-gorunur etkisini SAYISAL olarak dogruluyor.